# Lesson 3: Choosing the right assertion

This notebook goes with [Lesson 3 of *Unit Testing in Python*](https://acatlin.github.io/python-unit-testing/lessons/0003-choosing-assertions.html). The explanations are on that page. The code is here, ready to run.

- Run the cells from top to bottom, one at a time: click a cell and press Shift+Enter.
- This is your own copy. Change anything you like; nothing you do here alters the course. To keep your work, choose File, then Save a copy in Drive.

## Set up

Colab already has the pandas the lesson page was written with, version 2.2.3. Its pytest is older than the page's, so the next cell installs pytest 9.2.0. Then what you see here matches the page. It takes a few seconds.

In [ ]:
%pip install -q "pandas==2.2.3" "pytest==9.2.0"

Check what you have. A line that starts with `!` goes to the computer's command line, not to Python: it is this notebook's stand-in for the terminal.

In [ ]:
!python --version
!python -m pytest --version
!python -c "import pandas; print('pandas', pandas.__version__)"

## How files and commands work here

The lesson has you save files in an editor and type commands in a terminal. A notebook has neither, so two stand-ins do the job.

- **`%%writefile name`** on the first line of a cell saves the rest of the cell as a file, in place of running it. To change the file, edit the cell and run it again. (You can also open the file from the folder icon at the left edge of Colab.)
- **`!`** at the start of a line sends it to the command line. Every `!` line starts in the notebook's own folder, so the commands below begin with `cd cleaning-project &&` to step into the project first.

**What will look different from the lesson page.** The page shows output captured on Windows with Python 3.13. Colab runs Linux. So the `platform` line says `linux` and may name a slightly different Python, `rootdir:` is a folder under `/content`, paths are written with `/` in place of `\`, the rules of `=` signs may be wider or narrower, and the timings differ. The results are the same: the dots and `F`s, the lines marked `E`, and the counts on the last line.

## The project so far

This is the project as Lesson 2 left it: `clean.py`, and a test file with four tests (the three you were given and the one you wrote for `"$3,000"`). You should see `4 passed`.

In [ ]:
import os

os.makedirs("cleaning-project/tests", exist_ok=True)

In [ ]:
%%writefile cleaning-project/clean.py
"""Data-cleaning helpers: the starting point for Lesson 2.

Each function is small on purpose: small functions are easy to test.
"""

import re
import warnings

import pandas as pd


def rescale01(x):
    """Rescale a numeric column so its smallest value is 0 and its largest is 1."""
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo)


def clean_names(names):
    """Turn messy column names into lower-case snake_case."""
    cleaned = []
    for name in names:
        name = re.sub(r"[^a-z0-9]+", "_", name.lower())
        cleaned.append(name.strip("_"))
    return cleaned


def parse_money(x):
    """Convert a column of strings such as "$1,200.50" to numbers."""
    digits = x.str.replace(r"[$,]", "", regex=True)
    numbers = pd.to_numeric(digits, errors="coerce").astype("float64")
    n_bad = int(numbers.isna().sum() - x.isna().sum())
    if n_bad > 0:
        warnings.warn(f"{n_bad} value(s) could not be parsed as money", stacklevel=2)
    return numbers


def summarise_by(data, group, value):
    """Count rows and average one column within each group."""
    return data.groupby(group, as_index=False).agg(
        n=(value, "size"),
        mean=(value, "mean"),
    )

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import pandas as pd

from clean import clean_names, parse_money, rescale01


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]


def test_parse_money_reads_thousands_without_cents():
    assert parse_money(pd.Series(["$3,000"])).tolist() == [3000]

In [ ]:
!cd cleaning-project && python -m pytest

## Why not `==` for everything?

Three ways to compare two columns, each in a test of its own, saved in a scratch file. The expected value is wrong on purpose, so **all three are meant to fail**. Compare what each report tells you: a crash (`ValueError`), a bare `False`, and a message that says which value differs.

In [ ]:
%%writefile cleaning-project/tests/test_demo.py
import pandas as pd

from clean import rescale01


def test_ambiguous():
    result = rescale01(pd.Series([1, 2, 3]))
    expected = pd.Series([0, 0.5, 2])      # wrong on purpose
    assert result == expected


def test_vague():
    result = rescale01(pd.Series([1, 2, 3]))
    expected = pd.Series([0, 0.5, 2])      # wrong on purpose
    assert result.equals(expected)


def test_specific():
    result = rescale01(pd.Series([1, 2, 3]))
    expected = pd.Series([0, 0.5, 2])      # wrong on purpose
    pd.testing.assert_series_equal(result, expected)

In [ ]:
!cd cleaning-project && python -m pytest tests/test_demo.py

That file has done its job. Delete it.

In [ ]:
os.remove("cleaning-project/tests/test_demo.py")

## Testing a function that returns a data frame

`summarise_by()` is already in `clean.py`. Here is the test file with its two tests added at the end, and `summarise_by` added to the import line. You should see `6 passed`.

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import pandas as pd

from clean import clean_names, parse_money, rescale01, summarise_by


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]


def test_parse_money_reads_thousands_without_cents():
    assert parse_money(pd.Series(["$3,000"])).tolist() == [3000]


def test_summarise_by_counts_rows_and_averages_within_each_group():
    staff = pd.DataFrame({
        "dept": ["a", "a", "b"],
        "pay": [10, 20, 60],
    })
    expected = pd.DataFrame({
        "dept": ["a", "b"],
        "n": [2, 1],
        "mean": [15.0, 60.0],
    })
    pd.testing.assert_frame_equal(summarise_by(staff, "dept", "pay"), expected)


def test_summarise_by_returns_a_data_frame_with_the_documented_columns():
    staff = pd.DataFrame({"dept": ["a", "a", "b"], "pay": [10, 20, 60]})
    out = summarise_by(staff, "dept", "pay")
    assert isinstance(out, pd.DataFrame)
    assert list(out.columns) == ["dept", "n", "mean"]

In [ ]:
!cd cleaning-project && python -m pytest

## See the failure for yourself

In the cell that saves the test file, change the expected mean for "b" from `60.0` to `50.0`. Run that cell, then run the tests below. pandas names the column and the row that disagree. Put `60.0` back afterwards.

In [ ]:
!cd cleaning-project && python -m pytest

## Try it

Step 1 of the lesson is done.

2. In the expected data frame, change `"mean": [15.0, 60.0]` to `"mean": [15, 60]` (whole numbers in place of decimals). Save the file and run the tests. Does `assert_frame_equal()` still pass? Read the message. Now add the argument `check_dtype=False` to the call and run again. Put the decimals back when you have finished.

In [ ]:
!cd cleaning-project && python -m pytest

3. Write a new test for `rescale01()` with three assertions: the result has dtype `"float64"`, has the same length as the input, and its largest value is at most 1.

To add a test without retyping the file, start a cell with `%%writefile -a cleaning-project/tests/test_clean.py`. The `-a` means append: the cell is added to the end of the file. Leave two blank lines under that first line, and run the cell once only (running it twice adds the test twice).

In [ ]:
# step 3: your new test

In [ ]:
!cd cleaning-project && python -m pytest

Now go back to the [lesson page](https://acatlin.github.io/python-unit-testing/lessons/0003-choosing-assertions.html) for the quiz. The Lesson 4 notebook starts from this project with that seventh test already added.